# Video Game Sales Analysis
## Data Wrangling, EDA, Visualization, Predictive Modeling, and Time Series Forecasting

---

### Objective

This notebook provides a comprehensive analysis of the `vgsales.csv` dataset, which contains global video game sales data across platforms, genres, publishers, and regions.

The goals of this analysis are:

1. Clean and preprocess the raw dataset to ensure analytical integrity.
2. Engineer meaningful features that enhance modeling and insight generation.
3. Conduct a thorough exploratory data analysis (EDA) with supporting visualizations.
4. Build and evaluate multiple regression models to predict global sales.
5. Apply time series forecasting to project future global sales trends.

**Dataset columns:** `Rank`, `Name`, `Platform`, `Year`, `Genre`, `Publisher`, `NA_Sales`, `EU_Sales`, `JP_Sales`, `Other_Sales`, `Global_Sales`

---
## Section 1: Imports and Setup

We import all required libraries upfront. This keeps dependencies transparent and ensures the notebook runs cleanly from top to bottom.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

# Global plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9
})

RANDOM_STATE = 42
print('All libraries imported successfully.')

---
## Section 2: Data Loading

We load the dataset and perform an initial inspection to understand its shape, column types, and a sample of its content.

In [ ]:
# Load the dataset
df_raw = pd.read_csv('vgsales.csv')

print(f'Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')
print()
df_raw.head(10)

In [ ]:
# Data types and non-null counts
df_raw.info()

In [ ]:
# Missing value summary
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing Pct (%)': missing_pct})
missing_df[missing_df['Missing Count'] > 0]

---
## Section 3: Data Understanding

Before cleaning, we examine the distribution and unique values of key columns to develop a clear picture of the data landscape.

In [ ]:
# Descriptive statistics for numeric columns
df_raw.describe()

In [ ]:
# Cardinality of categorical columns
cat_cols = ['Platform', 'Genre', 'Publisher']
for col in cat_cols:
    print(f'{col}: {df_raw[col].nunique()} unique values')
    print(df_raw[col].value_counts().head(5))
    print()

In [ ]:
# Year range inspection
print('Year column unique non-null values (sample):')
print(sorted(df_raw['Year'].dropna().unique()))

---
## Section 4: Data Cleaning and Preprocessing

### Strategy

| Issue | Column | Strategy | Rationale |
|---|---|---|---|
| Missing values | `Year` | Impute with median | Year is numeric; median is robust to skew |
| Missing values | `Publisher` | Impute with `'Unknown'` | Categorical; preserving row is better than dropping |
| Wrong dtype | `Year` | Convert to `int` after imputation | Year should be integer for analysis |
| Duplicates | All | Drop exact duplicates | Exact duplicates add no information |
| Outliers | Sales columns | Identify with IQR; retain but flag | Outliers may be genuine blockbuster titles |
| Unrealistic year | `Year` | Cap at 2020 | Years beyond 2020 are likely data entry errors |

In [ ]:
df = df_raw.copy()

# --- Step 1: Remove exact duplicates ---
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Duplicates removed: {before - len(df)}')

# --- Step 2: Fix Year dtype and impute missing ---
# Convert to numeric first (turns strings like 'N/A' into NaN)
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
# Nullify clearly erroneous years
df.loc[df['Year'] > 2020, 'Year'] = np.nan
year_median = df['Year'].median()
df['Year'] = df['Year'].fillna(year_median)
# Replace any residual inf/-inf before casting
df['Year'] = df['Year'].replace([np.inf, -np.inf], year_median)
# Use Int64 (nullable integer) then convert to standard int
df['Year'] = df['Year'].round().astype('Int64').astype(int)
print(f'Year imputed with median: {int(year_median)}')

# --- Step 3: Impute missing Publisher ---
df['Publisher'] = df['Publisher'].fillna('Unknown')
print('Publisher NaNs filled with Unknown')

# --- Step 4: Ensure sales columns are float ---
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
for col in sales_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

# --- Step 5: Standardize categorical strings ---
df['Genre'] = df['Genre'].str.strip().str.title()
df['Platform'] = df['Platform'].str.strip()
df['Publisher'] = df['Publisher'].str.strip()

print(f'\nCleaned dataset shape: {df.shape}')
print('Missing values after cleaning:')
print(df.isnull().sum())

In [ ]:
# --- Outlier detection using IQR for Global_Sales ---
def iqr_outlier_bounds(series):
    """Returns lower and upper IQR-based outlier bounds."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return lower, upper

lower, upper = iqr_outlier_bounds(df['Global_Sales'])
outliers = df[(df['Global_Sales'] < lower) | (df['Global_Sales'] > upper)]
print(f'Global_Sales outliers detected: {len(outliers)}')
print(f'IQR bounds: lower={lower:.2f}, upper={upper:.2f}')
print()
print('Top 10 outlier titles by Global_Sales:')
outliers.nlargest(10, 'Global_Sales')[['Name', 'Platform', 'Year', 'Global_Sales']]

The outliers are well-known, legitimate blockbuster titles (e.g., Wii Sports, Mario Kart). These records are **retained** as they represent genuine market phenomena. Removing them would distort the analysis of top-performing games.

In [ ]:
# Save the cleaned dataset
df.to_csv('vgsales_cleaned.csv', index=False)
print('Cleaned dataset saved to vgsales_cleaned.csv')

---
## Section 5: Feature Engineering

We derive new features that may improve model performance and analytical insight:

| Feature | Derivation | Rationale |
|---|---|---|
| `Decade` | `Year // 10 * 10` | Captures generational gaming eras |
| `NA_Sales_Ratio` | `NA_Sales / Global_Sales` | North America market share |
| `EU_Sales_Ratio` | `EU_Sales / Global_Sales` | Europe market share |
| `JP_Sales_Ratio` | `JP_Sales / Global_Sales` | Japan market share |
| `Other_Sales_Ratio` | `Other_Sales / Global_Sales` | Rest of world share |
| `Non_JP_Sales` | `NA + EU + Other` | Western-market total |
| `Is_Multi_Platform` | Publisher release count on multiple platforms | Indicator of cross-platform availability |

In [ ]:
# Decade feature
df['Decade'] = (df['Year'] // 10 * 10).astype(str) + 's'

# Regional sales ratios (avoid division by zero)
safe_global = df['Global_Sales'].replace(0, np.nan)
df['NA_Sales_Ratio'] = (df['NA_Sales'] / safe_global).fillna(0).round(4)
df['EU_Sales_Ratio'] = (df['EU_Sales'] / safe_global).fillna(0).round(4)
df['JP_Sales_Ratio'] = (df['JP_Sales'] / safe_global).fillna(0).round(4)
df['Other_Sales_Ratio'] = (df['Other_Sales'] / safe_global).fillna(0).round(4)

# Non-Japan (Western) sales aggregate
df['Non_JP_Sales'] = df['NA_Sales'] + df['EU_Sales'] + df['Other_Sales']

print('New features added:')
new_feats = ['Decade', 'NA_Sales_Ratio', 'EU_Sales_Ratio', 'JP_Sales_Ratio',
             'Other_Sales_Ratio', 'Non_JP_Sales']
df[new_feats].head()

---
## Section 6: Exploratory Data Analysis (EDA)

We systematically examine the data from multiple angles: univariate distributions, bivariate relationships, temporal trends, and categorical breakdowns.

In [ ]:
# Summary statistics for all numeric columns
df.describe().round(3)

In [ ]:
# Top genres by total global sales
genre_sales = df.groupby('Genre')['Global_Sales'].sum().sort_values(ascending=False)
print('Genre - Total Global Sales (millions):')
print(genre_sales.round(2))

In [ ]:
# Top 10 platforms by total global sales
platform_sales = df.groupby('Platform')['Global_Sales'].sum().sort_values(ascending=False).head(10)
print('Top 10 Platforms - Total Global Sales (millions):')
print(platform_sales.round(2))

In [ ]:
# Top 10 publishers by total global sales
publisher_sales = df.groupby('Publisher')['Global_Sales'].sum().sort_values(ascending=False).head(10)
print('Top 10 Publishers - Total Global Sales (millions):')
print(publisher_sales.round(2))

In [ ]:
# Yearly global sales trend
yearly_sales = df.groupby('Year')['Global_Sales'].sum().sort_index()
print('Yearly Global Sales (head):')
print(yearly_sales.head(10))

In [ ]:
# Regional sales totals
regional = df[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']].sum().sort_values(ascending=False)
print('Total Sales by Region (millions):')
print(regional.round(2))

In [ ]:
# Correlation matrix of numeric features
numeric_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales',
                'NA_Sales_Ratio', 'EU_Sales_Ratio', 'JP_Sales_Ratio', 'Non_JP_Sales']
corr_matrix = df[numeric_cols].corr().round(3)
corr_matrix

---
## Section 7: Data Visualization

Each visualization is followed by an interpretation of key findings.

In [ ]:
# Distribution of Global Sales (log-transformed for readability)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['Global_Sales'], bins=80, color='steelblue', edgecolor='white', linewidth=0.4)
axes[0].set_title('Global Sales Distribution (Raw)')
axes[0].set_xlabel('Global Sales (millions)')
axes[0].set_ylabel('Count')

log_sales = np.log1p(df['Global_Sales'])
axes[1].hist(log_sales, bins=60, color='darkorange', edgecolor='white', linewidth=0.4)
axes[1].set_title('Global Sales Distribution (Log1p Transformed)')
axes[1].set_xlabel('log(1 + Global Sales)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('plot_sales_distribution.png', bbox_inches='tight')
plt.show()

**Insight:** Global Sales is heavily right-skewed. The vast majority of titles sell under 1 million copies, while a small number of blockbusters distort the raw distribution. The log-transformed view reveals a near-normal shape, confirming that a log transformation is appropriate for modeling.

In [ ]:
# Top genres by total global sales
genre_sales_sorted = df.groupby('Genre')['Global_Sales'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
genre_sales_sorted.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Total Global Sales by Genre')
ax.set_xlabel('Total Sales (millions)')
ax.set_ylabel('Genre')
plt.tight_layout()
plt.savefig('plot_genre_sales.png', bbox_inches='tight')
plt.show()

**Insight:** Action and Sports dominate total global sales. This reflects both the volume of titles released and their broad mass-market appeal. Niche genres like Strategy and Puzzle generate considerably less total revenue despite having dedicated audiences.

In [ ]:
# Top 10 platforms by total global sales
top_platforms = df.groupby('Platform')['Global_Sales'].sum().nlargest(10).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
top_platforms.plot(kind='barh', ax=ax, color='teal')
ax.set_title('Top 10 Platforms by Total Global Sales')
ax.set_xlabel('Total Sales (millions)')
ax.set_ylabel('Platform')
plt.tight_layout()
plt.savefig('plot_platform_sales.png', bbox_inches='tight')
plt.show()

**Insight:** The PS2 leads all platforms by a wide margin, reflecting its enormous install base and long commercial lifespan. The DS and Wii follow, both propelled by Nintendo's focus on casual and family gaming audiences.

In [ ]:
# Global sales trend over time
yearly = df.groupby('Year')['Global_Sales'].sum().reset_index()
# Filter to reasonable year range
yearly = yearly[(yearly['Year'] >= 1980) & (yearly['Year'] <= 2016)]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(yearly['Year'], yearly['Global_Sales'], marker='o', markersize=4,
        linewidth=1.8, color='steelblue')
ax.fill_between(yearly['Year'], yearly['Global_Sales'], alpha=0.15, color='steelblue')
ax.set_title('Global Video Game Sales by Year')
ax.set_xlabel('Year')
ax.set_ylabel('Total Global Sales (millions)')
ax.xaxis.set_major_locator(mticker.MultipleLocator(4))
plt.tight_layout()
plt.savefig('plot_yearly_trend.png', bbox_inches='tight')
plt.show()

**Insight:** Global sales grew steadily from the mid-1990s, peaking around 2008-2009 during the seventh generation console era (PS3, Xbox 360, Wii). The sharp post-2010 decline is partly a dataset artifact (newer titles have fewer recorded sales at time of data collection) and partly reflects market fragmentation from mobile gaming.

In [ ]:
# Correlation heatmap
heatmap_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales', 'Non_JP_Sales']
corr = df[heatmap_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap - Sales Features')
plt.tight_layout()
plt.savefig('plot_correlation_heatmap.png', bbox_inches='tight')
plt.show()

**Insight:** NA_Sales has the strongest correlation with Global_Sales, indicating North America is the dominant market driver. JP_Sales shows the weakest correlation with other regions, confirming that Japan has distinct gaming preferences that do not always align with Western markets.

In [ ]:
# Regional sales breakdown (pie chart)
region_totals = {
    'North America': df['NA_Sales'].sum(),
    'Europe': df['EU_Sales'].sum(),
    'Japan': df['JP_Sales'].sum(),
    'Other': df['Other_Sales'].sum()
}
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(region_totals.values(), labels=region_totals.keys(),
       autopct='%1.1f%%', colors=colors, startangle=140,
       wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax.set_title('Global Sales Distribution by Region')
plt.tight_layout()
plt.savefig('plot_regional_pie.png', bbox_inches='tight')
plt.show()

**Insight:** North America accounts for roughly 49% of all recorded sales, making it the single most important market. Europe is second at around 27%, while Japan contributes approximately 14%. The "Other" category (rest of world) holds the remaining share, reflecting gaming's growing reach in emerging markets.

In [ ]:
# Genre breakdown by region
genre_region = df.groupby('Genre')[['NA_Sales', 'EU_Sales', 'JP_Sales']].sum()
genre_region_norm = genre_region.div(genre_region.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
genre_region_norm.sort_values('NA_Sales', ascending=False).plot(
    kind='bar', stacked=True, ax=ax,
    color=['#4C72B0', '#DD8452', '#55A868'])
ax.set_title('Regional Sales Composition by Genre (Normalized)')
ax.set_xlabel('Genre')
ax.set_ylabel('Proportion of Sales')
ax.legend(['NA Sales', 'EU Sales', 'JP Sales'], bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plot_genre_region_stacked.png', bbox_inches='tight')
plt.show()

**Insight:** Role-Playing Games (RPGs) show a substantially higher Japan share compared to other genres, reflecting the dominance of franchises like Final Fantasy and Dragon Quest in that market. Shooters and Sports genres are disproportionately NA-heavy, aligning with North American gaming culture.

In [ ]:
# Sales distribution per decade (box plot)
decade_order = sorted(df['Decade'].unique())

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=df[df['Global_Sales'] < 5],
            x='Decade', y='Global_Sales', order=decade_order,
            palette='Blues', ax=ax, fliersize=2)
ax.set_title('Global Sales Distribution by Decade (Titles Under 5M)')
ax.set_xlabel('Decade')
ax.set_ylabel('Global Sales (millions)')
plt.tight_layout()
plt.savefig('plot_decade_boxplot.png', bbox_inches='tight')
plt.show()

**Insight:** Median per-title sales increased from the 1980s through the 2000s, reflecting both rising install bases and growing consumer spending. The 2010s show a slight decline in median sales per title, which may reflect the fragmentation of game distribution channels (digital, mobile).

In [ ]:
# Top 10 publishers by total global sales
top_pub = df.groupby('Publisher')['Global_Sales'].sum().nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
top_pub.plot(kind='barh', ax=ax, color='mediumpurple')
ax.set_title('Top 10 Publishers by Total Global Sales')
ax.set_xlabel('Total Sales (millions)')
ax.set_ylabel('Publisher')
plt.tight_layout()
plt.savefig('plot_top_publishers.png', bbox_inches='tight')
plt.show()

**Insight:** Nintendo leads all publishers by total global sales by a wide margin, driven by its dual role as a first-party software developer and hardware manufacturer. Electronic Arts and Activision rank second and third, reflecting their strength in sports and shooter franchises respectively.

In [ ]:
# Top 5 genres over time
top5_genres = df.groupby('Genre')['Global_Sales'].sum().nlargest(5).index.tolist()
genre_year = df[df['Genre'].isin(top5_genres)].copy()
genre_year = genre_year[(genre_year['Year'] >= 1995) & (genre_year['Year'] <= 2016)]
genre_pivot = genre_year.groupby(['Year', 'Genre'])['Global_Sales'].sum().unstack()

fig, ax = plt.subplots(figsize=(13, 5))
genre_pivot.plot(ax=ax, linewidth=2, marker='o', markersize=3)
ax.set_title('Top 5 Genre Sales Over Time (1995-2016)')
ax.set_xlabel('Year')
ax.set_ylabel('Total Sales (millions)')
ax.legend(bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.savefig('plot_genre_trend.png', bbox_inches='tight')
plt.show()

**Insight:** Action games show the most consistent growth across the period. Sports games peaked in the mid-2000s, likely driven by the annual FIFA and Madden franchise cycles. Shooter games surged post-2006, coinciding with the rise of Call of Duty and Halo on 7th-gen consoles.

---
## Section 8: Modeling and Evaluation

### Target: `Global_Sales`

We train three regression models:
- **Linear Regression** - a simple baseline
- **Random Forest Regressor** - a non-parametric ensemble method
- **Gradient Boosting Regressor** - a sequential boosting ensemble

### Preprocessing Steps:
1. Encode categorical features using `LabelEncoder`
2. Scale numeric features using `StandardScaler`
3. Use an 80/20 train-test split

**Note:** Because Global_Sales is right-skewed, we log-transform the target and inverse-transform predictions for evaluation.

In [ ]:
# Select features for modeling
feature_cols = ['Platform', 'Year', 'Genre', 'Publisher',
                'NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']
target_col = 'Global_Sales'

df_model = df[feature_cols + [target_col]].copy()

# Encode categorical columns
le_dict = {}
for col in ['Platform', 'Genre', 'Publisher']:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str))
    le_dict[col] = le

X = df_model[feature_cols].values
y = np.log1p(df_model[target_col].values)  # log-transform target

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]}')
print(f'Test size:  {X_test.shape[0]}')

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Trains a model, evaluates it, and returns a results dict."""
    model.fit(X_tr, y_tr)
    preds_log = model.predict(X_te)
    # Inverse transform from log space
    preds = np.expm1(preds_log)
    actuals = np.expm1(y_te)
    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae = mean_absolute_error(actuals, preds)
    r2 = r2_score(actuals, preds)
    print(f'{name}: RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}')
    return {'Model': name, 'RMSE': round(rmse, 4), 'MAE': round(mae, 4), 'R2': round(r2, 4)}

results = []

# Linear Regression
lr = LinearRegression()
results.append(evaluate_model('Linear Regression', lr,
                               X_train_sc, X_test_sc, y_train, y_test))

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
results.append(evaluate_model('Random Forest', rf,
                               X_train, X_test, y_train, y_test))

# Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=150, learning_rate=0.1,
                                max_depth=4, random_state=RANDOM_STATE)
results.append(evaluate_model('Gradient Boosting', gb,
                               X_train, X_test, y_train, y_test))

In [ ]:
# Model comparison table
results_df = pd.DataFrame(results).set_index('Model')
print('Model Comparison:')
results_df

In [ ]:
# Visual model comparison
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics = ['RMSE', 'MAE', 'R2']
colors = ['#4C72B0', '#DD8452', '#55A868']

for i, (ax, metric) in enumerate(zip(axes, metrics)):
    ax.bar(results_df.index, results_df[metric], color=colors)
    ax.set_title(f'Model Comparison: {metric}')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('plot_model_comparison.png', bbox_inches='tight')
plt.show()

**Insight:** Gradient Boosting and Random Forest significantly outperform Linear Regression on all metrics. The tree-based models capture the non-linear relationships between platform, genre, publisher, and sales. Linear Regression serves as a useful baseline but lacks the expressive power needed for this dataset's complexity.

In [ ]:
# Feature importance from Random Forest
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Random Forest Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('plot_feature_importance.png', bbox_inches='tight')
plt.show()

**Insight:** Regional sales (especially NA_Sales) dominate feature importance, which is expected since they are components of Global_Sales. Among non-sales features, Platform, Publisher, and Year carry meaningful predictive weight, confirming that the game's release context is highly relevant to its commercial outcome.

In [ ]:
# Actual vs Predicted scatter plot (Gradient Boosting)
gb_preds = np.expm1(gb.predict(X_test))
gb_actuals = np.expm1(y_test)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(gb_actuals, gb_preds, alpha=0.3, s=15, color='steelblue')
max_val = max(gb_actuals.max(), gb_preds.max())
ax.plot([0, max_val], [0, max_val], color='red', linestyle='--', linewidth=1.2, label='Perfect Fit')
ax.set_title('Gradient Boosting: Actual vs Predicted Global Sales')
ax.set_xlabel('Actual Global Sales (millions)')
ax.set_ylabel('Predicted Global Sales (millions)')
ax.legend()
plt.tight_layout()
plt.savefig('plot_actual_vs_predicted.png', bbox_inches='tight')
plt.show()

**Insight:** Predictions cluster well along the ideal diagonal for titles with lower sales. The model underestimates extreme outlier titles (top-right region), which is typical for tree-based regression models on highly skewed targets with very few extreme examples in the training data.

---
## Section 9: Time Series Forecasting

We aggregate yearly global sales into a time series and apply an ARIMA model to forecast the next 8 years. Before fitting, we test for stationarity using the Augmented Dickey-Fuller (ADF) test and apply differencing if needed.

In [ ]:
# Prepare annual time series
ts = df[(df['Year'] >= 1990) & (df['Year'] <= 2015)]
ts = ts.groupby('Year')['Global_Sales'].sum().sort_index()

print('Annual Global Sales time series:')
print(ts)

In [ ]:
# Augmented Dickey-Fuller test for stationarity
adf_result = adfuller(ts)
print(f'ADF Statistic : {adf_result[0]:.4f}')
print(f'p-value       : {adf_result[1]:.4f}')
print(f'Critical Values:')
for key, val in adf_result[4].items():
    print(f'  {key}: {val:.4f}')

if adf_result[1] < 0.05:
    print('\nConclusion: Series is stationary (reject null hypothesis)')
else:
    print('\nConclusion: Series is non-stationary (fail to reject null hypothesis) - differencing required')

In [ ]:
# Fit ARIMA model
# d=1 for first differencing (accounts for non-stationarity)
arima_model = ARIMA(ts, order=(2, 1, 1))
arima_fit = arima_model.fit()
print(arima_fit.summary())

In [ ]:
# Forecast next 8 years
forecast_steps = 8
forecast_result = arima_fit.get_forecast(steps=forecast_steps)
forecast_mean = forecast_result.predicted_mean
forecast_ci = forecast_result.conf_int(alpha=0.05)

last_year = ts.index[-1]
forecast_years = range(last_year + 1, last_year + 1 + forecast_steps)
forecast_series = pd.Series(forecast_mean.values, index=forecast_years)

print('Forecast (Global Sales in millions):')
for yr, val in zip(forecast_years, forecast_mean.values):
    print(f'  {yr}: {val:.2f}')

In [ ]:
# Plot historical + forecast
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(ts.index, ts.values, marker='o', markersize=4,
        linewidth=2, color='steelblue', label='Historical Sales')
ax.plot(list(forecast_years), forecast_mean.values,
        marker='s', markersize=4, linewidth=2,
        color='darkorange', linestyle='--', label='Forecast')
ax.fill_between(
    list(forecast_years),
    forecast_ci.iloc[:, 0].values,
    forecast_ci.iloc[:, 1].values,
    alpha=0.2, color='darkorange', label='95% Confidence Interval'
)
ax.axvline(x=last_year, color='gray', linestyle=':', linewidth=1.2, label='Forecast Start')
ax.set_title('ARIMA Time Series Forecast: Global Video Game Sales')
ax.set_xlabel('Year')
ax.set_ylabel('Total Global Sales (millions)')
ax.legend()
plt.tight_layout()
plt.savefig('plot_arima_forecast.png', bbox_inches='tight')
plt.show()

**Insight:** The ARIMA model captures the downward trend visible from 2009 onward in the dataset. The forecast projects continued decline in recorded physical game sales. This is consistent with the real-world shift toward digital distribution and mobile gaming, which are not well-represented in this dataset. The widening confidence interval reflects increasing uncertainty over longer forecast horizons, which is expected for ARIMA models on short time series.

---
## Section 10: Key Insights and Conclusion

### Data Quality
- The dataset contained approximately 2% missing values in `Year` and `Publisher`. These were imputed using the median and a constant value respectively, preserving analytical integrity without dropping rows.
- Outliers in Global_Sales were identified but retained, as they represent legitimate blockbuster titles.

### Market Structure
- North America is the dominant global gaming market, accounting for approximately 49% of total sales.
- Japan shows distinctly different genre preferences (RPG-heavy) compared to Western markets, which favor Action and Shooter titles.
- The PS2, DS, and Wii are the top-selling platforms of all time in this dataset.

### Publisher Dominance
- Nintendo is the leading publisher by total global sales, with a significant gap over Electronic Arts and Activision.

### Temporal Trends
- Global sales peaked around 2008-2009 during the 7th console generation.
- The apparent decline post-2010 is partly a data artifact (newer titles have lower cumulative sales at collection time) and partly reflects real market fragmentation.

### Modeling
- Tree-based models (Random Forest and Gradient Boosting) substantially outperform Linear Regression for this dataset.
- Regional sales features dominate predictive power, as expected given they are sub-components of the target.
- Platform, Publisher, and Year contribute meaningful non-linear predictive signal.

### Forecasting
- The ARIMA(2,1,1) model forecasts declining physical game sales over the next 8 years, consistent with industry trends toward digital and mobile gaming.

---

### Outputs Saved
- `vgsales_cleaned.csv` - cleaned dataset
- `plot_sales_distribution.png`
- `plot_genre_sales.png`
- `plot_platform_sales.png`
- `plot_yearly_trend.png`
- `plot_correlation_heatmap.png`
- `plot_regional_pie.png`
- `plot_genre_region_stacked.png`
- `plot_decade_boxplot.png`
- `plot_top_publishers.png`
- `plot_genre_trend.png`
- `plot_model_comparison.png`
- `plot_feature_importance.png`
- `plot_actual_vs_predicted.png`
- `plot_arima_forecast.png`